**1. Importing the dependencies**

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

In [4]:
df = pd.read_csv("../data/processed/autism_processed.csv")

In [5]:
df.shape

(800, 20)

In [6]:
df.columns

Index(['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score',
       'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'age', 'gender',
       'ethnicity', 'jaundice', 'austim', 'contry_of_res', 'used_app_before',
       'result', 'relation', 'Class/ASD'],
      dtype='str')

In [7]:
X = df.drop(columns=["Class/ASD"])
y = df["Class/ASD"]

In [8]:
print(X)

     A1_Score  A2_Score  A3_Score  A4_Score  A5_Score  A6_Score  A7_Score  \
0           1         0         1         0         1         0         1   
1           0         0         0         0         0         0         0   
2           1         1         1         1         1         1         1   
3           0         0         0         0         0         0         0   
4           0         0         0         0         0         0         0   
..        ...       ...       ...       ...       ...       ...       ...   
795         0         1         0         0         0         0         0   
796         0         1         1         0         0         1         0   
797         0         0         0         0         0         0         0   
798         0         0         0         0         0         0         0   
799         0         1         0         0         0         0         0   

     A8_Score  A9_Score  A10_Score   age  gender  ethnicity  jaundice  aust

In [9]:
print(y)

0      0
1      0
2      1
3      0
4      0
      ..
795    0
796    0
797    0
798    0
799    0
Name: Class/ASD, Length: 800, dtype: int64


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
print(y_train.shape)
print(y_test.shape)

(640,)
(160,)


In [12]:
y_train.value_counts()

Class/ASD
0    515
1    125
Name: count, dtype: int64

In [13]:
y_test.value_counts()

Class/ASD
0    124
1     36
Name: count, dtype: int64

**SMOTE (Synthetic Minority Oversampling technique)**

In [16]:
from imblearn.over_sampling import SMOTE

In [17]:
smote = SMOTE(random_state=42)

In [18]:
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [19]:
print(y_train_smote.shape)

(1030,)


In [20]:
print(y_train_smote.value_counts())

Class/ASD
1    515
0    515
Name: count, dtype: int64


In [21]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression

In [22]:
# dictionary of classifiers
models = {
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "Linear Regression": LinearRegression()
}

In [23]:
# dictionary to store the cross validation results
cv_scores = {}

# perform 5-fold cross validation for each model
for model_name, model in models.items():
  print(f"Training {model_name} with default parameters...")
  if model_name == "Linear Regression":
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=5, scoring="neg_mean_squared_error")
    cv_scores[model_name] = -scores  # make positive
    print(f"{model_name} Cross-Validation MSE: {np.mean(-scores):.2f}")
  else:
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=5, scoring="accuracy")
    cv_scores[model_name] = scores
    print(f"{model_name} Cross-Validation Accuracy: {np.mean(scores):.2f}")
  print("-"*50)

Training KNN with default parameters...
KNN Cross-Validation Accuracy: 0.86
--------------------------------------------------
Training SVM with default parameters...
SVM Cross-Validation Accuracy: 0.82
--------------------------------------------------
Training Decision Tree with default parameters...
Decision Tree Cross-Validation Accuracy: 0.86
--------------------------------------------------
Training Random Forest with default parameters...
Random Forest Cross-Validation Accuracy: 0.92
--------------------------------------------------
Training XGBoost with default parameters...
XGBoost Cross-Validation Accuracy: 0.90
--------------------------------------------------
Training Linear Regression with default parameters...
Linear Regression Cross-Validation MSE: 0.14
--------------------------------------------------


In [24]:
cv_scores

{'KNN': array([0.86893204, 0.84951456, 0.86407767, 0.85436893, 0.8592233 ]),
 'SVM': array([0.7961165 , 0.83980583, 0.84951456, 0.82038835, 0.7961165 ]),
 'Decision Tree': array([0.7961165 , 0.87864078, 0.87378641, 0.8592233 , 0.87378641]),
 'Random Forest': array([0.90776699, 0.92718447, 0.9223301 , 0.91747573, 0.9223301 ]),
 'XGBoost': array([0.87378641, 0.9223301 , 0.89320388, 0.91262136, 0.91747573]),
 'Linear Regression': array([0.13668041, 0.15472216, 0.16441731, 0.13502933, 0.11919318])}

In [25]:
# Initializing models
knn = KNeighborsClassifier()
svm = SVC(random_state=42)
decision_tree = DecisionTreeClassifier(random_state=42)
random_forest = RandomForestClassifier(random_state=42)
xgboost_classifier = XGBClassifier(random_state=42)
linear_regression = LinearRegression()

In [26]:
# Hyperparameter grids for RandomizedSearchCV

param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

param_grid_svm = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1]
}

param_grid_dt = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 10, 20, 30, 50, 70],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

param_grid_rf = {
    "n_estimators": [50, 100, 200, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

param_grid_xgb = {
    "n_estimators": [50, 100, 200, 500],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.1, 0.2, 0.3],
    "subsample": [0.5, 0.7, 1.0],
    "colsample_bytree": [0.5, 0.7, 1.0]
}

param_grid_lr = {
    "fit_intercept": [True, False]
}

In [28]:
# perform RandomizedSearchCV for each model
random_search_knn = RandomizedSearchCV(estimator=knn, param_distributions=param_grid_knn, n_iter=10, cv=5, scoring="accuracy", random_state=42)
random_search_svm = RandomizedSearchCV(estimator=svm, param_distributions=param_grid_svm, n_iter=10, cv=5, scoring="accuracy", random_state=42)
random_search_dt = RandomizedSearchCV(estimator=decision_tree, param_distributions=param_grid_dt, n_iter=20, cv=5, scoring="accuracy", random_state=42)
random_search_rf = RandomizedSearchCV(estimator=random_forest, param_distributions=param_grid_rf, n_iter=20, cv=5, scoring="accuracy", random_state=42)
random_search_xgb = RandomizedSearchCV(estimator=xgboost_classifier, param_distributions=param_grid_xgb, n_iter=20, cv=5, scoring="accuracy", random_state=42)
random_search_lr = RandomizedSearchCV(estimator=linear_regression, param_distributions=param_grid_lr, n_iter=2, cv=5, scoring="neg_mean_squared_error", random_state=42)

In [30]:
# Get the model with best score
best_model = None
best_score = -float('inf')

if random_search_knn.best_score_ > best_score:
  best_model = random_search_knn.best_estimator_
  best_score = random_search_knn.best_score_

if random_search_svm.best_score_ > best_score:
  best_model = random_search_svm.best_estimator_
  best_score = random_search_svm.best_score_

if random_search_dt.best_score_ > best_score:
  best_model = random_search_dt.best_estimator_
  best_score = random_search_dt.best_score_

if random_search_rf.best_score_ > best_score:
  best_model = random_search_rf.best_estimator_
  best_score = random_search_rf.best_score_

if random_search_xgb.best_score_ > best_score:
  best_model = random_search_xgb.best_estimator_
  best_score = random_search_xgb.best_score_

if random_search_lr.best_score_ > best_score:
  best_model = random_search_lr.best_estimator_
  best_score = random_search_lr.best_score_

print(f"Best Model: {best_model}")
print(f"Best Score: {best_score}")

Best Model: RandomForestClassifier(bootstrap=False, max_depth=20, n_estimators=50,
                       random_state=42)
Best Score: 0.9271844660194175


In [31]:
print(f"Best Model: {best_model}")
print(f"Best Cross-Validation Accuracy: {best_score:.2f}")

Best Model: RandomForestClassifier(bootstrap=False, max_depth=20, n_estimators=50,
                       random_state=42)
Best Cross-Validation Accuracy: 0.93


In [32]:
# save the best model
with open("../models/best_model.pkl", "wb") as f:
  pickle.dump(best_model, f)

In [34]:
# evaluate on test data
y_test_pred = best_model.predict(X_test)

print("="*60)
print("EVALUATION OF BEST MODEL ON TEST DATA")
print("="*60)
print(f"Accuracy Score: {accuracy_score(y_test, y_test_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))
print("="*60)

EVALUATION OF BEST MODEL ON TEST DATA
Accuracy Score: 0.8187

Confusion Matrix:
[[108  16]
 [ 13  23]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       124
           1       0.59      0.64      0.61        36

    accuracy                           0.82       160
   macro avg       0.74      0.75      0.75       160
weighted avg       0.82      0.82      0.82       160

